<a href="https://colab.research.google.com/github/LarissaGiovanna/agente-cineData/blob/main/agente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Conexão banco de dados

In [1]:
import sqlite3
from pathlib import Path

db_path = (Path.cwd() / "../src/cinedata/data/cinerocket.db").resolve()

if not db_path.exists():
    raise FileNotFoundError(f"Banco não encontrado em: {db_path}")

conn = sqlite3.connect(str(db_path))

tabelas = [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
)]

for t in tabelas:
    print(f"\n== {t} ==")
    for cid, nome, tipo, notnull, default, pk in conn.execute(f"PRAGMA table_info({t})"):
        print(f"  {nome} ({tipo}){' [PK]' if pk else ''}")
    total = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  -> {total} linhas")
    cur = conn.execute(f"SELECT * FROM {t} LIMIT 3")
    print("  colunas:", [d[0] for d in cur.description])
    for linha in cur.fetchall():
        print("   ", linha)


== alembic_version ==
  version_num (VARCHAR(32)) [PK]
  -> 1 linhas
  colunas: ['version_num']
    ('0001_initial_schema',)

== bridge_movie_company ==
  sk_movie_id (VARCHAR(64)) [PK]
  sk_company_id (VARCHAR(64)) [PK]
  -> 116326 linhas
  colunas: ['sk_movie_id', 'sk_company_id']
    ('8a85beff7b0f64692b227466b2c9c4abb274cf1225da4d5ce495da2180ba89c8', 'a93d6d918018600b5fba9d39dbb7898430ece8d5fb29e42a2f8db59a37f5faf7')
    ('c1e53d82aeac91f16040913257678ba690813daa7a5c5d3e182866a339553ce0', 'f0ed91f80406391e38f2e7cc647c452f8f5919e770d7d1247969c27f14bc0980')
    ('ceeda10da2fb7789a0c63741e562f050cec19f48ab727861678a10e51f9fc1f9', 'e8170686c11431164d3ad11ef91043bfb5a67a6a4af9f7e9f6e3ea51476398bd')

== bridge_movie_genre ==
  sk_movie_id (VARCHAR(64)) [PK]
  sk_genre_id (VARCHAR(64)) [PK]
  -> 121521 linhas
  colunas: ['sk_movie_id', 'sk_genre_id']
    ('b5c312c4d8cee94972412b9ea785976f9dbb2c26dd11480d9f2b9b4691df1571', '271a22f75d73e13fa558d991442ecde8d75a34a0b9f6216d44b116f8612c82dc'

In [2]:
def consultar(sql):
    cur = conn.execute(sql)
    print([d[0] for d in cur.description])
    for linha in cur.fetchall():
        print("  ", linha)
    print()

checagens = [
    "SELECT tipo_pessoa, COUNT(*) FROM dim_people GROUP BY tipo_pessoa",
    "SELECT status_filme, COUNT(*) FROM dim_movies GROUP BY status_filme",
    "SELECT MIN(ano_lancamento), MAX(ano_lancamento) FROM dim_movies",
    """SELECT SUM(receita_usd IS NULL), SUM(receita_usd = 0),
              SUM(orcamento_usd IS NULL), SUM(orcamento_usd = 0),
              SUM(nota_imdb IS NULL), SUM(popularidade IS NULL)
       FROM fact_movies_performance""",
    "SELECT SUM(receita_usd IS NULL AND lucro_usd <> 0) FROM fact_movies_performance",
    "SELECT MIN(nota_imdb), MAX(nota_imdb), MIN(nota_tmdb), MAX(nota_tmdb) FROM fact_movies_performance",
    """SELECT MIN(nota_media_usuarios), MAX(nota_media_usuarios),
              MIN(qtd_avaliacoes_usuarios), MAX(qtd_avaliacoes_usuarios) FROM dim_reviews""",
    "SELECT COUNT(*), COUNT(DISTINCT sk_movie_id) FROM dim_reviews",
    "SELECT MIN(rating), MAX(rating), COUNT(DISTINCT sk_movie_id) FROM movie_reviews",
]

for sql in checagens:
    consultar(sql)

['tipo_pessoa', 'COUNT(*)']
   ('Ator', 273400)
   ('Diretor', 65200)
   ('Roteirista', 86056)

['status_filme', 'COUNT(*)']
   ('Em Produção', 598)
   ('Lançado', 94304)
   ('Planejado', 50)
   ('Pós-Produção', 693)

['MIN(ano_lancamento)', 'MAX(ano_lancamento)']
   (2016, 2029)

['SUM(receita_usd IS NULL)', 'SUM(receita_usd = 0)', 'SUM(orcamento_usd IS NULL)', 'SUM(orcamento_usd = 0)', 'SUM(nota_imdb IS NULL)', 'SUM(popularidade IS NULL)']
   (92272, 0, 87719, 0, 12674, 3615)

['SUM(receita_usd IS NULL AND lucro_usd <> 0)']
   (6296,)

['MIN(nota_imdb)', 'MAX(nota_imdb)', 'MIN(nota_tmdb)', 'MAX(nota_tmdb)']
   (0.0, 10.0, 0.0, 10.0)

['MIN(nota_media_usuarios)', 'MAX(nota_media_usuarios)', 'MIN(qtd_avaliacoes_usuarios)', 'MAX(qtd_avaliacoes_usuarios)']
   (0.0, 10.0, 1, 13)

['COUNT(*)', 'COUNT(DISTINCT sk_movie_id)']
   (40267, 40267)

['MIN(rating)', 'MAX(rating)', 'COUNT(DISTINCT sk_movie_id)']
   (0.0, 10.0, 40267)



Exemplos de prompts:
- "

In [3]:
sqls_referencia = {
  "1_lucro_medio_por_genero": """
    SELECT g.nome_genero,
           ROUND(AVG(f.lucro_brl), 2) AS lucro_medio_brl,
           COUNT(*) AS qtd_filmes
    FROM fact_movies_performance f
    JOIN bridge_movie_genre bg ON bg.sk_movie_id = f.sk_movie_id
    JOIN dim_genres g ON g.sk_genre_id = bg.sk_genre_id
    WHERE f.receita_brl IS NOT NULL
    GROUP BY g.nome_genero
    ORDER BY lucro_medio_brl DESC
  """,
  "2_maior_margem_de_lucro": """
    SELECT m.titulo, f.receita_brl, f.orcamento_brl, f.lucro_brl,
           ROUND(f.lucro_brl * 1.0 / f.receita_brl, 4) AS margem
    FROM fact_movies_performance f
    JOIN dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.receita_brl IS NOT NULL
      AND f.orcamento_brl IS NOT NULL
      AND f.receita_brl > 0
    ORDER BY margem DESC
    LIMIT 10
  """,
  "3_dupla_ator_diretor": """
    SELECT a.nome_pessoa AS ator, d.nome_pessoa AS diretor, COUNT(*) AS qtd_filmes
    FROM bridge_movie_person bpa
    JOIN dim_people a ON a.sk_person_id = bpa.sk_person_id AND a.tipo_pessoa = 'Ator'
    JOIN bridge_movie_person bpd ON bpd.sk_movie_id = bpa.sk_movie_id
    JOIN dim_people d ON d.sk_person_id = bpd.sk_person_id AND d.tipo_pessoa = 'Diretor'
    GROUP BY a.sk_person_id, d.sk_person_id
    ORDER BY qtd_filmes DESC
    LIMIT 5
  """,
  "4_ator_mais_participacoes_ultimos_5_anos": """
    SELECT p.nome_pessoa, COUNT(*) AS participacoes
    FROM dim_movies m
    JOIN bridge_movie_person bp ON bp.sk_movie_id = m.sk_movie_id
    JOIN dim_people p ON p.sk_person_id = bp.sk_person_id AND p.tipo_pessoa = 'Ator'
    WHERE m.data_lancamento >= date('now', 'start of year', '-4 years')
      AND m.data_lancamento <= date('now')
    GROUP BY p.sk_person_id
    ORDER BY participacoes DESC
    LIMIT 5
  """,
  "5_filmes_mais_avaliados_pelos_usuarios": """
    SELECT m.titulo, r.qtd_avaliacoes_usuarios, r.nota_media_usuarios
    FROM dim_reviews r
    JOIN dim_movies m ON m.sk_movie_id = r.sk_movie_id
    ORDER BY r.qtd_avaliacoes_usuarios DESC,
             r.nota_media_usuarios DESC,
             m.titulo
    LIMIT 10
  """,
}

for nome, sql in sqls_referencia.items():
    print("###", nome)
    consultar(sql)

### 1_lucro_medio_por_genero
['nome_genero', 'lucro_medio_brl', 'qtd_filmes']
   ('Science Fiction', 520783718.33, 227)
   ('Adventure', 514678791.39, 393)
   ('Action', 343223021.87, 594)
   ('Fantasy', 335448446.35, 255)
   ('Family', 324648758.33, 264)
   ('Animation', 303517897.37, 239)
   ('War', 194016703.28, 91)
   ('History', 146826836.61, 171)
   ('Comedy', 143589472.55, 813)
   ('Mystery', 119462433.49, 246)
   ('Music', 111985444.28, 100)
   ('Romance', 98645875.99, 312)
   ('Crime', 93988989.27, 294)
   ('Thriller', 92872345.47, 623)
   ('Horror', 92805277.2, 383)
   ('Drama', 84161852.72, 1297)
   ('Tv Movie', 9228474.48, 10)
   ('Documentary', 5028959.44, 153)
   ('Western', -3581373.25, 20)

### 2_maior_margem_de_lucro
['titulo', 'receita_brl', 'orcamento_brl', 'lucro_brl', 'margem']
   ("Dad, I'm Sorry", 95303762.5, 712.12, 95303050.39, 1.0)
   ('Etlb', 3225800, 161.29, 3225638.71, 1.0)
   ('Jailbait', 23778840.8, 1688.44, 23777152.36, 0.9999)
   ('Trivikrama', 52328, 2

Prompt

In [4]:
from cinedata.prompt import montar_prompt
p = montar_prompt("2026-10-05")
print("data ok:" , "2026-10-05" in p)
print("sem placeholder:", "__DATA_HOJE__" not in p)
print(p[:300])

data ok: True
sem placeholder: True
Você é o analista de dados do CineData Analytics. Responda em português, consultando
o banco SQLite com a ferramenta de SQL.

ESCOPO E SEGURANÇA
- Responda só sobre o catálogo de filmes (filmes, elenco e equipe, gêneros, produtoras,
  receita, notas, avaliações). Para outro assunto, responda apenas:
